In [1]:
pip install pandas scikit-learn matplotlib

In [2]:
import numpy as np
import pandas as pd

# 1. Data load
df = pd.read_csv("/content/netflix.csv")
print(df.shape)

# 2. Missing value fill
for col in ["director", "country", "listed_in", "rating", "type"]:
    df[col] = df[col].fillna("").astype(str)

# 3. Cleaning function
def clean_list(text):
    return " ".join([t.strip().replace(" ", "").lower() for t in text.split(",") if t.strip()])

# 4. Column cleaning
df["genres_c"]   = df["listed_in"].apply(clean_list)
df["director_c"] = df["director"].apply(clean_list)
df["country_c"]  = df["country"].apply(clean_list)
df["type_c"]     = df["type"].str.replace(" ", "").str.lower()
df["rating_c"]   = df["rating"].str.replace("-", "").str.lower()

# 5. Soup adding
df["soup"] = (
    df["type_c"] + " " +
    df["genres_c"] + " " + df["genres_c"] + " " +
    df["director_c"] + " " +
    df["country_c"] + " " +
    df["rating_c"]
)

df[["title", "soup"]].head()

(8790, 10)


,title,soup
0,Dick Johnson Is Dead,movie documentaries documentaries kirstenjohns...
1,Ganglands,tvshow crimetvshows internationaltvshows tvact...
2,Midnight Mass,tvshow tvdramas tvhorror tvmysteries tvdramas ...
3,Confessions of an Invisible Girl,movie children&familymovies comedies children&...
4,Sankofa,movie dramas independentmovies internationalmo...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()          # soup convert into number
tfidf_matrix = tfidf.fit_transform(df["soup"])

print(tfidf_matrix.shape)

(8790, 5403)


In [4]:
from sklearn.metrics.pairwise import linear_kernel

# all title calculate each other similar
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print(cosine_sim.shape)

#  lookup table
indices = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()
print(indices.head())

(8790, 8790)
title
dick johnson is dead                0
ganglands                           1
midnight mass                       2
confessions of an invisible girl    3
sankofa                             4
dtype: int64


In [5]:
def recommend(title, n=10):
    key = title.lower()
    if key not in indices:
        return f"'{title}'"

    idx = indices[key]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:n+1]

    result = df.iloc[[i for i, _ in scores]][["title", "type", "listed_in", "director", "country"]].copy()
    result["similarity"] = [round(s, 3) for _, s in scores]
    return result

recommend("Ganglands")


,title,type,listed_in,director,country,similarity
6738,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,France,0.874
8408,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,France,0.874
576,Fatal Destiny,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Pakistan,0.810
8334,Agent Raghav,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,India,0.803
7972,Smoking,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Japan,0.803
390,The Eagle of El-Se'eed,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Pakistan,0.793
486,The Truth,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Pakistan,0.793
7946,Monkey Twins,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Thailand,0.783
8450,Filinta,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Turkey,0.775
8519,Kaçak,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Not Given,Turkey,0.775


In [6]:
recommend("Midnight Mass")
recommend("Sankofa")

,title,type,listed_in,director,country,similarity
280,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",Dibakar Banerjee,India,0.530
4739,Dagh Ujala,Movie,"Dramas, Independent Movies, International Movies",Raj Amit Kumar,United States,0.524
6379,Traffic Signal,Movie,"Dramas, Independent Movies, International Movies",Madhur Bhandarkar,India,0.522
3356,Udaan,Movie,"Dramas, Independent Movies, International Movies",Vikramaditya Motwane,India,0.518
3382,Garbage,Movie,"Dramas, Independent Movies, International Movies",Qaushiq Mukherjee,India,0.518
5399,Mandi,Movie,"Dramas, Independent Movies, International Movies",Shyam Benegal,India,0.518
1871,Michael,Movie,"Dramas, Independent Movies, International Movies",Ribhu Dasgupta,India,0.512
5115,I Am,Movie,"Dramas, Independent Movies, International Movies",Onir,India,0.512
5943,Starred Up,Movie,"Dramas, Independent Movies, International Movies",David Mackenzie,United Kingdom,0.507
941,Tottaa Pataaka Item Maal,Movie,"Dramas, Independent Movies, International Movies",Aditya Kripalani,India,0.505


In [7]:
def genre_set(i):
    return set(g.strip() for g in df.loc[i, "listed_in"].split(","))

def genre_precision_at_k(k=10, sample=300, seed=42):
    rng = np.random.default_rng(seed)
    scores_list = []
    for idx in rng.choice(len(df), sample, replace=False):
        top = np.argsort(cosine_sim[idx])[::-1][1:k+1]
        base = genre_set(idx)
        hits = [len(base & genre_set(j)) > 0 for j in top]
        scores_list.append(np.mean(hits))
    return np.mean(scores_list)

print("Genre precision@10:", round(genre_precision_at_k(), 3))

Genre precision@10: 0.997


In [8]:
# not include  genre  soup
df["soup_nogenre"] = (
    df["type_c"] + " " +
    df["director_c"] + " " +
    df["country_c"] + " " +
    df["rating_c"]
)

tfidf2 = TfidfVectorizer()
m2 = tfidf2.fit_transform(df["soup_nogenre"])
cosine_sim_ng = linear_kernel(m2, m2)

def genre_precision_ng(k=10, sample=300, seed=42):
    rng = np.random.default_rng(seed)
    out = []
    for idx in rng.choice(len(df), sample, replace=False):
        top = np.argsort(cosine_sim_ng[idx])[::-1][1:k+1]
        base = genre_set(idx)
        out.append(np.mean([len(base & genre_set(j)) > 0 for j in top]))
    return np.mean(out)

print("Genre precision@10 ( not include genre):", round(genre_precision_ng(), 3))

Genre precision@10 ( not include genre): 0.457
